# 02 · Training EfficientDet-Lite0 (BiFPN) — VannameScan

Transfer learning dari bobot COCO (TF Hub `efficientdet/lite0/feature-vector/1`) untuk 3 kelas
`besar`, `sedang`, `kecil` dengan **TFLite Model Maker 0.4.3** (keputusan D-010).

**PERLU DIJALANKAN DI COLAB** (Runtime → Change runtime type → **GPU**). Tidak ada angka hasil di notebook ini
sampai kamu menjalankannya.

Cara pakai:
1. Ubah **hanya** sel *Konfigurasi*.
2. *Runtime → Run all*. Notebook berhenti dengan pesan jelas bila audit dataset menemukan masalah yang memblokir.
3. Bila Colab terputus: sambungkan ulang lalu *Run all* lagi. Training **melanjutkan** dari checkpoint terakhir
   di Google Drive. Jangan mengubah `EPOCHS`, `BATCH_SIZE`, atau `LEARNING_RATE` di tengah eksperimen, karena
   jadwal learning rate dihitung dari total langkah.

Yang **tidak** dikerjakan di sini (sesi berikutnya): evaluasi test/mAP@0.5 per kelas, ekspor TFLite INT8,
`evaluate_counting`.

## 1. Konfigurasi

In [ ]:
# === KONFIGURASI — satu-satunya sel yang boleh diubah ===
DRIVE_ROOT = "/content/drive/MyDrive/udang_VannameScan"
DATASET_DIR = f"{DRIVE_ROOT}/dataset_udang"   # berisi train/ valid/ test/ (Pascal VOC dari Roboflow)
AUDIT_DIR = f"{DRIVE_ROOT}/audit"              # keluaran ml/01_prepare_dataset.py
RUNS_DIR = f"{DRIVE_ROOT}/runs"                # checkpoint + TensorBoard per eksperimen
HUB_CACHE_DIR = f"{DRIVE_ROOT}/tfhub_cache"    # bobot COCO diunduh sekali lalu dipakai ulang
# Bobot COCO EfficientDet-Lite0 (BiFPN). Bila tfhub.dev gagal, isi dengan path folder hasil ekstrak manual.
HUB_URI = "https://tfhub.dev/tensorflow/efficientdet/lite0/feature-vector/1"

REPO_URL = "https://github.com/dhimasss/VannameScan.git"
REPO_BRANCH = "main"
PY39_ENV = "/content/py39"                     # lingkungan Python 3.9 terpisah (dibuat ulang tiap sesi)

SEED = 42
IMAGE_SIZE = 320          # PRD D.7 input_size
EPOCHS = 50
BATCH_SIZE = 8
LEARNING_RATE = None      # None = default Model Maker (0.08 × BATCH_SIZE / 64)
DETERMINISTIC = True      # False bila GPU melaporkan op tanpa implementasi deterministik

# Augmentasi bawaan toolchain. Dataset sudah diaugmentasi di Roboflow, jadi hanya flip horizontal.
# Scale jitter dimatikan (1.0, 1.0): kelas ditentukan oleh ukuran udang di citra, dan jitter mengubah ukuran itu.
INPUT_RAND_HFLIP = True
JITTER_MIN, JITTER_MAX = 1.0, 1.0

MAX_INSTANCES_PER_IMAGE = 100   # batas ground-truth per gambar saat training (skrip menolak bila terlampaui)
TFLITE_MAX_DETECTIONS = 100     # PRD D.7: minimal 100
MAP_FREQ = 5                    # hitung mAP validasi tiap N epoch (TensorBoard)
KEEP_CHECKPOINTS = 3

# Eksperimen: nilai di sini menimpa nilai global di atas. Contoh variasi, bukan rekomendasi hasil.
EXPERIMENTS = {
    "exp1_head_only":   dict(train_whole_model=False),
    "exp2_whole_model": dict(train_whole_model=True),
    "exp3_whole_lr004": dict(train_whole_model=True, learning_rate=0.04),
}
RUN_EXPERIMENTS = ["exp1_head_only"]   # eksperimen yang dijalankan pada sesi ini (berurutan)

# Audit: lanjut training walau ada kebocoran antar split / augmentasi di test. Ubah ke True hanya setelah diputuskan.
ACCEPT_AUDIT_WARNINGS = False

## 2. Google Drive dan repo

In [ ]:
import os, subprocess
from google.colab import drive

drive.mount("/content/drive")
REPO_DIR = "/content/VannameScan"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
for required in ("ml/train_efficientdet.py", "ml/run_py39.sh", "ml/requirements-train.txt", "ml/01_prepare_dataset.py"):
    assert os.path.exists(f"{REPO_DIR}/{required}"), f"{required} tidak ada di branch {REPO_BRANCH}"
assert os.path.isdir(DATASET_DIR), f"Dataset tidak ditemukan: {DATASET_DIR}"
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)

## 3. Lingkungan Python 3.9 (TFLite Model Maker)

Model Maker 0.4.3 tidak terpasang di Python 3.12 bawaan Colab. Sel ini membuat lingkungan terpisah dengan `uv`
dan memasang `ml/requirements-train.txt` (TensorFlow 2.8.4 + CUDA 11.8/cuDNN 8.6 dari pip). ±5–10 menit per sesi.

In [ ]:
import os, subprocess
if not os.path.exists(f"{PY39_ENV}/.ready"):
    subprocess.run("apt-get -qq install -y libusb-1.0-0 > /dev/null", shell=True, check=True)  # dibutuhkan tflite-support
    subprocess.run(["pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "venv", "-q", "-p", "3.9", PY39_ENV], check=True)
    subprocess.run(["uv", "pip", "install", "-q", "-p", f"{PY39_ENV}/bin/python",
                    "-r", f"{REPO_DIR}/ml/requirements-train.txt"], check=True)
    open(f"{PY39_ENV}/.ready", "w").close()
os.environ["VANNAMESCAN_PY39_ENV"] = PY39_ENV
os.environ["TFHUB_CACHE_DIR"] = HUB_CACHE_DIR
RUN_PY39 = f"{REPO_DIR}/ml/run_py39.sh"
print("Lingkungan siap:", PY39_ENV)

## 4. Uji lingkungan: GPU, Model Maker, bobot COCO

In [ ]:
check = f"""
import tensorflow as tf, tensorflow_hub as hub
from tflite_model_maker import object_detector
gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow", tf.__version__, "| GPU:", gpus)
assert gpus, "GPU tidak terdeteksi oleh TensorFlow 2.8 — periksa runtime GPU dan log di atas"
hub.KerasLayer("{HUB_URI}")
print("Bobot COCO EfficientDet-Lite0 termuat")
"""
subprocess.run([RUN_PY39, "-c", check], check=True)

## 5. Audit dataset (read-only)

Menjalankan `ml/01_prepare_dataset.py`. Training **dihentikan** bila ada masalah yang membuat Model Maker gagal
atau salah membaca data (`mm_*`, label tidak dikenal/salah huruf, XML rusak, kotak degenerasi). Kebocoran antar split
dan augmentasi di test juga menghentikan notebook kecuali `ACCEPT_AUDIT_WARNINGS = True`.

In [ ]:
import json
subprocess.run(["python", f"{REPO_DIR}/ml/01_prepare_dataset.py", "--dataset", DATASET_DIR,
                "--out", AUDIT_DIR, "--seed", str(SEED)], check=True, stdout=subprocess.DEVNULL)
with open(f"{AUDIT_DIR}/report.json", encoding="utf-8") as handle:
    audit = json.load(handle)

blocking = {k: n for k, n in audit["issues_by_kind"].items()
            if k.startswith("mm_") or k in ("label_unknown", "label_case", "xml_unreadable", "box_degenerate")}
warnings = {"kelompok bocor antar split": len(audit["group_leakage"]),
            "kelompok >1 gambar di test": len(audit["test_groups_with_multiple_images"])}
print("Jumlah per split:", json.dumps(audit["counts"], indent=1))
print("Masalah anotasi:", audit["issues_by_kind"])
print("Peringatan:", warnings)
if blocking:
    raise RuntimeError(f"Masalah pemblokir: {blocking}. Lihat {AUDIT_DIR}/issues.csv")
if any(warnings.values()) and not ACCEPT_AUDIT_WARNINGS:
    raise RuntimeError(f"Peringatan audit {warnings}. Putuskan dulu (lihat {AUDIT_DIR}/report.md), "
                       "lalu set ACCEPT_AUDIT_WARNINGS = True bila tetap lanjut.")

## 6. Training (resumable)

Setiap eksperimen menulis ke `RUNS_DIR/<nama>`: `ckpt-<epoch>` (3 terakhir), log TensorBoard, `run_config.json`,
`val_metrics.json`, dan `DONE.json` saat selesai. Eksperimen yang sudah punya `DONE.json` dilewati.

In [ ]:
os.makedirs(RUNS_DIR, exist_ok=True)
base = dict(dataset_dir=DATASET_DIR, runs_dir=RUNS_DIR, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED,
            image_size=IMAGE_SIZE, learning_rate=LEARNING_RATE, input_rand_hflip=INPUT_RAND_HFLIP,
            jitter_min=JITTER_MIN, jitter_max=JITTER_MAX, max_instances_per_image=MAX_INSTANCES_PER_IMAGE,
            tflite_max_detections=TFLITE_MAX_DETECTIONS, map_freq=MAP_FREQ, keep_checkpoints=KEEP_CHECKPOINTS,
            deterministic=DETERMINISTIC, hub_uri=HUB_URI)
for name in RUN_EXPERIMENTS:
    config = {**base, **EXPERIMENTS[name], "run_name": name, "cache_dir": f"/content/tfrecord_cache/{name}"}
    config_path = f"/content/{name}.json"
    with open(config_path, "w", encoding="utf-8") as handle:
        json.dump(config, handle, indent=2)
    print(f"=== {name} ===")
    subprocess.run([RUN_PY39, f"{REPO_DIR}/ml/train_efficientdet.py", "--config", config_path], check=True)

## 7. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "$RUNS_DIR"

## 8. Ringkasan dari berkas hasil

Dibaca dari `DONE.json` tiap eksperimen (hasil eksekusi nyata). Metrik = COCO pada split **valid**:
`AP` = mAP@0.5:0.95, `AP50` = mAP@0.5, `AP_/<kelas>` = AP@0.5:0.95 per kelas.

In [ ]:
import pandas as pd
rows = []
for name, overrides in EXPERIMENTS.items():
    done = f"{RUNS_DIR}/{name}/DONE.json"
    if not os.path.exists(done):
        rows.append({"eksperimen": name, "status": "belum selesai"})
        continue
    with open(done, encoding="utf-8") as handle:
        result = json.load(handle)
    with open(f"{RUNS_DIR}/{name}/run_config.json", encoding="utf-8") as handle:
        used = json.load(handle)["config"]
    metrics = result["val_metrics"]
    rows.append({"eksperimen": name, "status": "selesai", "train_whole_model": used["train_whole_model"],
                 "epochs": used["epochs"], "batch": used["batch_size"], "lr": used["learning_rate"],
                 "val mAP@0.5": metrics["AP50"], "val mAP@0.5:0.95": metrics["AP"],
                 **{f"AP {c}": metrics[f"AP_/{c}"] for c in ("besar", "sedang", "kecil")},
                 "detik (sesi terakhir)": result["session_seconds"]})
pd.DataFrame(rows)

## 9. Tabel hasil eksperimen (diisi dari sel 8 — jangan diisi sebelum ada run nyata)

| Eksperimen | train_whole_model | Epoch | Batch | LR | val mAP@0.5 | val mAP@0.5:0.95 | AP besar | AP sedang | AP kecil | Waktu training | Catatan |
|---|---|---|---|---|---|---|---|---|---|---|---|
| exp1_head_only | | | | | | | | | | | |
| exp2_whole_model | | | | | | | | | | | |
| exp3_whole_lr004 | | | | | | | | | | | |

Model yang dipilih untuk ekspor (sesi berikutnya) ditentukan dari **validasi**, bukan test.

## Catatan augmentasi (didokumentasikan apa adanya)

- Offline: augmentasi Roboflow yang sudah ada di train (tidak ditambah).
- Online (Model Maker/EfficientDet `InputReader`): flip horizontal acak (`input_rand_hflip`), scale jitter
  **dimatikan** (`jitter_min = jitter_max = 1.0`), `autoaugment_policy = None`, `grid_mask = False`.